# Entrainement Et Evaluation LLM Support

Ce notebook charge les jeux de donnees prepares, evalue un LLM open-weight de base, le personnalise, puis compare les deux versions sur le jeu de test.


## Installer Les Dependances

Installez les dependances du projet dans l'environnement actif du notebook avant d'executer le reste du notebook.


In [ ]:
%pip install -r requirements.txt




## Imports

Ajoute les bibliotheques necessaires pour charger les jeux de donnees, executer le modele et calculer les metriques d'evaluation.


In [3]:
import json
import pandas as pd
import torch
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from support_classification.prompts import build_prompt
from support_classification.paths import LABELS_PATH, PROJECT_ROOT, TRAIN_DATASET_PATH, TEST_DATASET_PATH
from support_classification.sequence_classification import (
    WeightedClassificationTrainer,
    build_class_weights,
    build_compute_metrics,
    build_label_mappings,
    dataframe_to_hf_dataset,
    encode_classification_dataframe,
    evaluate_sequence_classifier,
    save_label_mappings,
    tokenize_classification_dataset,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments

# Affiche les textes de prompt longs sans troncature lors de l'inspection des jeux de donnees.
pd.set_option('display.max_colwidth', 120)





ModuleNotFoundError: No module named 'torch'

## Charger Les Jeux De Donnees Prepares

Charge les fichiers CSV train et test produits par le notebook de preparation.


In [3]:
train_dataset = pd.read_csv(TRAIN_DATASET_PATH)
test_dataset = pd.read_csv(TEST_DATASET_PATH)
valid_labels = json.loads(LABELS_PATH.read_text(encoding='utf-8'))['labels']

print('Train shape:', train_dataset.shape)
print('Test shape:', test_dataset.shape)
print('Valid labels:', valid_labels)




Train shape: (479, 2)
Test shape: (120, 2)


## Charger Le Modele De Classification

Charge un modele sequence classification base sur Qwen2.5 et prepare les mappings de labels pour un apprentissage supervise direct.


In [ ]:
# Choix initial de modele conserve ici comme option de repli.
# Identifiant officiel correct : mistralai/Ministral-3-8B-Instruct-2512
# Ce modele instruct open-weight est multilingue et suffisamment solide pour la classification de tickets de support,
# il reste donc une bonne solution de secours si le plus petit modele n'atteint pas le score cible plus tard.
# base_model_id = 'mistralai/Ministral-3-8B-Instruct-2512'

# Commence avec un modele instruct plus petit car le jeu d'entrainement est modeste (479 exemples).
# Qwen2.5-1.5B-Instruct reste suffisamment fort pour une classification multilingue tout en restant raisonnable a fine-tuner.
base_model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

label_to_id, id_to_label = build_label_mappings(valid_labels)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    base_model_id,
    num_labels=len(valid_labels),
    id2label=id_to_label,
    label2id=label_to_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# Conserve une adaptation LoRA legere pour limiter le cout memoire tout en laissant la tete de classification s'entrainer.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    modules_to_save=['score'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print('Loaded model:', base_model_id)
print('Number of labels:', len(valid_labels))






## Inspecter Un Exemple De Prompt

Le dataset prepare conserve un champ `prompt`. Le classifieur apprend directement a predire un label a partir de ce texte, sans generation libre.


In [ ]:
# Les helpers de prompt sont importes depuis support_classification.prompts afin que le notebook d'entrainement utilise exactement
# le meme template que le notebook de preparation des donnees.
# Les jeux de donnees traites contiennent deja ce format de prompt, et cet exemple montre comment
# le reconstruire plus tard pour de nouveaux tickets au moment de l'inference.
example_prompt = build_prompt(
    subject='Compatibility Inquiry',
    body='I recently purchased a MacBook Air M1 and need to know if it is compatible with my external wireless monitor setup.',
    language='en',
    business_type='Tech Online Store',
    valid_labels=valid_labels,
)

print(example_prompt)




You are a support ticket classification assistant.
Predict the ticket category from the information below.

Subject: Compatibility Inquiry
Body: I recently purchased a MacBook Air M1 and need to know if it is compatible with my external wireless monitor setup.
Language: en
Business type: Tech Online Store

Answer:


## Preparer Les Donnees De Classification

Encode les labels en identifiants entiers, tokenize les prompts et calcule des poids de classes pour attenuer le desequilibre du dataset.


In [ ]:
# Encode les labels en entiers pour la classification supervisee.
encoded_train_dataset = encode_classification_dataframe(train_dataset, label_to_id)
encoded_test_dataset = encode_classification_dataframe(test_dataset, label_to_id)

hf_train_dataset = dataframe_to_hf_dataset(encoded_train_dataset)
hf_test_dataset = dataframe_to_hf_dataset(encoded_test_dataset)

tokenized_train_dataset = tokenize_classification_dataset(hf_train_dataset, tokenizer, max_length=768)
tokenized_test_dataset = tokenize_classification_dataset(hf_test_dataset, tokenizer, max_length=768)

# Passe a False pour tester une loss non ponderee, ce qui peut parfois mieux optimiser le weighted F1.
use_class_weights = False

class_weights = None
if use_class_weights:
    class_weights = build_class_weights(encoded_train_dataset['labels'].to_numpy(), num_labels=len(valid_labels))

compute_metrics = build_compute_metrics(id_to_label)

display(encoded_train_dataset[['prompt', 'answer', 'labels']].head(2))
print(tokenized_train_dataset)
print('Use class weights:', use_class_weights)
print('Class weights:', class_weights)





## Fine-Tuner Le Classifieur

Entraine un modele de classification avec une tete supervisee et une loss ponderee pour mieux traiter les classes rares.


In [ ]:
# Sauvegarde le modele personnalise dans le projet pour pouvoir le reutiliser plus tard sans reentrainement complet.
adapter_output_dir = PROJECT_ROOT / 'artifacts' / 'qwen25-support-seq-cls-lora'
adapter_output_dir.mkdir(parents=True, exist_ok=True)

bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled

print('Evaluation during training: no')
print('Load best model at end: False')

# Active le gradient checkpointing pour reduire l'utilisation memoire pendant le fine-tuning.
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir=str(adapter_output_dir),
    num_train_epochs=6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='no',
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to='none',
    bf16=bf16_enabled,
    fp16=fp16_enabled,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    remove_unused_columns=True,
)

trainer = WeightedClassificationTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    processing_class=tokenizer,
    class_weights=class_weights,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(str(adapter_output_dir))
tokenizer.save_pretrained(str(adapter_output_dir))
save_label_mappings(adapter_output_dir / 'label_mappings.json', valid_labels, label_to_id)

print('Saved personalized classifier artifacts to:', adapter_output_dir)






## Verifier Les Artefacts Sauvegardes

Confirme que le modele entraine, le tokenizer et les mappings de labels ont bien ete sauvegardes.


In [ ]:
# Liste les fichiers sauvegardes pour confirmer que les artefacts du modele et du tokenizer sont disponibles.
saved_artifacts = sorted(path.name for path in adapter_output_dir.iterdir())
print('Saved artifacts:')
for artifact_name in saved_artifacts:
    print('-', artifact_name)





## Lancer Une Verification Qualitative

Recharge le modele entraine et inspecte quelques predictions sur des exemples du train pour verifier que la sortie est coherente.


In [ ]:
# Recharge le tokenizer et l'adapter entraine depuis le disque pour verifier que les artefacts sauvegardes sont reutilisables.
inference_tokenizer = AutoTokenizer.from_pretrained(str(adapter_output_dir))
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

base_model_for_inference = AutoModelForSequenceClassification.from_pretrained(
    base_model_id,
    num_labels=len(valid_labels),
    id2label=id_to_label,
    label2id=label_to_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)
personalized_model = PeftModel.from_pretrained(base_model_for_inference, str(adapter_output_dir))
personalized_model.config.pad_token_id = inference_tokenizer.pad_token_id
personalized_model.eval()

qualitative_examples = train_dataset.head(3).copy()
qualitative_examples, _, _ = evaluate_sequence_classifier(
    qualitative_examples,
    personalized_model,
    inference_tokenizer,
    id_to_label,
    'Personalized classifier qualitative check',
)
display(qualitative_examples[['prompt', 'answer', 'prediction']])






## Evaluer Les Deux Classifieurs

Mesure les scores `weighted F1` et `macro F1` sur le jeu de test pour le classifieur de base puis pour le classifieur entraine.

Le `weighted F1` est la metrique officielle de la specification. Le `macro F1` est conserve comme indicateur complementaire pour analyser le comportement du modele sur toutes les classes, y compris les plus rares.


In [ ]:
base_classifier = AutoModelForSequenceClassification.from_pretrained(
    base_model_id,
    num_labels=len(valid_labels),
    id2label=id_to_label,
    label2id=label_to_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)
base_classifier.config.pad_token_id = tokenizer.pad_token_id
base_classifier.eval()

base_test_predictions, base_weighted_f1, base_macro_f1 = evaluate_sequence_classifier(
    test_dataset,
    base_classifier,
    tokenizer,
    id_to_label,
    'Base classifier',
)

display(base_test_predictions[['prompt', 'answer', 'prediction']].head(5))





In [ ]:
personalized_test_predictions, personalized_weighted_f1, personalized_macro_f1 = evaluate_sequence_classifier(
    test_dataset,
    personalized_model,
    inference_tokenizer,
    id_to_label,
    'Personalized classifier',
)

display(personalized_test_predictions[['prompt', 'answer', 'prediction']].head(5))





## Comparer Au Score Cible

Resume le resultat final du classifieur personnalise par rapport a la cible du projet.


In [ ]:
target_f1 = 0.92

comparison_df = pd.DataFrame(
    [
        {
            'model': 'Base classifier',
            'weighted_f1': base_weighted_f1,
            'macro_f1': base_macro_f1,
            'target_92_reached': base_weighted_f1 >= target_f1,
        },
        {
            'model': 'Personalized classifier',
            'weighted_f1': personalized_weighted_f1,
            'macro_f1': personalized_macro_f1,
            'target_92_reached': personalized_weighted_f1 >= target_f1,
        },
    ]
)
comparison_df['weighted_f1'] = comparison_df['weighted_f1'].map(lambda score: f'{score:.2%}')
comparison_df['macro_f1'] = comparison_df['macro_f1'].map(lambda score: f'{score:.2%}')

print(f'Base classifier reached the 92% target: {base_weighted_f1 >= target_f1}')
print(f'Personalized classifier reached the 92% target: {personalized_weighted_f1 >= target_f1}')
display(comparison_df)



